# Box-prompt robustness analysis

This notebook analyzes the per-image JSONL produced by the COCO box-prompt experiment. It measures:

- **Mean change**: mean score over perturbed prompts minus the score from the original prompt. Negative values indicate degradation.
- **Prompt-induced variance**: sample variance (`ddof=1`) across the perturbed-prompt scores within an image. Larger values indicate greater sensitivity to prompt perturbations.

The notebook creates class-level plots and attribute-level plots using mask area and mask-to-box coverage.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"

## Configuration

Change `RESULTS_JSONL` if your results file has a different location. All figures are also saved as PNG files.

In [ ]:
RESULTS_JSONL = Path("./results/sam3_coco_box_robustness.jsonl")
FIGURE_DIR = Path("./robustness-plots")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Quantile bins used in attribute-level plots. With 100 images, 5 bins
# normally provide a useful balance between resolution and stability.
N_ATTRIBUTE_BINS = 5
MIN_CLASS_COUNT = 1

## Load JSONL and construct one row per image

The ground-truth mask area is taken from `original_prompt.metrics.ground_truth_area`. Box coverage is defined as:

$$\text{mask-to-box coverage} = \frac{\text{ground-truth mask pixels}}{\text{ground-truth box width} \times \text{ground-truth box height}}.$ $

In [ ]:
def load_image_level_results(path: Path) -> pd.DataFrame:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            original = record["original_prompt"]["metrics"]
            perturbed = record["perturbed_prompts"]
            pert_iou = np.asarray([p["metrics"]["iou"] for p in perturbed], dtype=float)
            pert_dice = np.asarray([p["metrics"]["dice"] for p in perturbed], dtype=float)
            if len(pert_iou) < 2:
                raise ValueError(f"Line {line_number} has fewer than two perturbations")

            _, _, box_width, box_height = map(float, record["ground_truth_box_xywh"])
            box_area = box_width * box_height
            mask_area = float(original["ground_truth_area"])

            records.append({
                "image_id": record["image_id"],
                "file_name": record["file_name"],
                "annotation_id": record["annotation_id"],
                "category_name": record["category_name"],
                "original_iou": float(original["iou"]),
                "original_dice": float(original["dice"]),
                "mean_perturbed_iou": pert_iou.mean(),
                "mean_perturbed_dice": pert_dice.mean(),
                "delta_iou": pert_iou.mean() - float(original["iou"]),
                "delta_dice": pert_dice.mean() - float(original["dice"]),
                "variance_iou": pert_iou.var(ddof=1),
                "variance_dice": pert_dice.var(ddof=1),
                "gt_mask_pixels": mask_area,
                "gt_box_area": box_area,
                "mask_box_coverage": mask_area / box_area if box_area > 0 else np.nan,
                "num_perturbations": len(pert_iou),
            })
    return pd.DataFrame.from_records(records)

df = load_image_level_results(RESULTS_JSONL)
print(f"Loaded {len(df)} images across {df['category_name'].nunique()} classes")
display(df.head())
display(df[["delta_iou", "delta_dice", "variance_iou", "variance_dice",
            "gt_mask_pixels", "mask_box_coverage"]].describe())

## Helper functions

Error bars are normal-approximation 95% confidence intervals for the mean (`1.96 × SEM`). For a small number of images per class, interpret them descriptively rather than as reliable inferential intervals.

In [ ]:
METRIC_COLORS = {"IoU": "#4C72B0", "Dice": "#DD8452"}

def mean_ci95(values):
    values = pd.Series(values).dropna().astype(float)
    mean = values.mean()
    if len(values) < 2:
        return mean, np.nan
    return mean, 1.96 * values.sem()

def class_summary(data: pd.DataFrame, columns: dict[str, str]) -> pd.DataFrame:
    rows = []
    for category, group in data.groupby("category_name", sort=False):
        if len(group) < MIN_CLASS_COUNT:
            continue
        for label, column in columns.items():
            mean, ci95 = mean_ci95(group[column])
            rows.append({"category_name": category, "metric": label,
                         "mean": mean, "ci95": ci95, "n": len(group)})
    return pd.DataFrame(rows)

def plot_class_results(data: pd.DataFrame, columns: dict[str, str], title: str,
                       xlabel: str, output_name: str, zero_line: bool = False):
    summary = class_summary(data, columns)
    order = (summary[summary["metric"] == "IoU"]
             .sort_values("mean")["category_name"].tolist())
    fig, axes = plt.subplots(1, 2, figsize=(14, max(5, 0.45 * len(order))), sharey=True)
    for ax, metric in zip(axes, ("IoU", "Dice")):
        part = summary[summary["metric"] == metric].set_index("category_name").loc[order]
        y = np.arange(len(order))
        ax.barh(y, part["mean"], xerr=part["ci95"], capsize=3,
                color=METRIC_COLORS[metric], alpha=0.85)
        ax.set_yticks(y, [f"{name} (n={int(part.loc[name, 'n'])})" for name in order])
        ax.set_title(metric)
        ax.set_xlabel(xlabel)
        if zero_line:
            ax.axvline(0, color="black", linewidth=1)
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / output_name, dpi=200)
    plt.show()
    return summary

def binned_attribute_summary(data: pd.DataFrame, attribute: str, columns: dict[str, str]):
    working = data.dropna(subset=[attribute]).copy()
    working["attribute_bin"] = pd.qcut(
        working[attribute], q=N_ATTRIBUTE_BINS, duplicates="drop"
    )
    rows = []
    for interval, group in working.groupby("attribute_bin", observed=True):
        for label, column in columns.items():
            mean, ci95 = mean_ci95(group[column])
            rows.append({
                "bin": str(interval), "metric": label, "mean": mean,
                "ci95": ci95, "x_median": group[attribute].median(), "n": len(group)
            })
    return pd.DataFrame(rows)

def plot_attribute_results(data: pd.DataFrame, attribute: str, columns: dict[str, str],
                           title: str, xlabel: str, ylabel: str, output_name: str,
                           log_x: bool = False, zero_line: bool = False):
    summary = binned_attribute_summary(data, attribute, columns)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, metric in zip(axes, ("IoU", "Dice")):
        part = summary[summary["metric"] == metric].sort_values("x_median")
        ax.errorbar(part["x_median"], part["mean"], yerr=part["ci95"],
                    marker="o", linewidth=2, capsize=4, color=METRIC_COLORS[metric])
        for _, row in part.iterrows():
            ax.annotate(f"n={int(row['n'])}", (row["x_median"], row["mean"]),
                        xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8)
        if log_x:
            ax.set_xscale("log")
        if zero_line:
            ax.axhline(0, color="black", linewidth=1)
        ax.set_title(metric)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / output_name, dpi=200)
    plt.show()
    return summary

## 1. Average change in IoU and Dice grouped by object class

Values below zero mean that perturbing the box reduced performance relative to the original prompt.

In [ ]:
class_change = plot_class_results(
    df,
    {"IoU": "delta_iou", "Dice": "delta_dice"},
    title="Mean performance change after box perturbation, by object class",
    xlabel="Mean perturbed score − original score",
    output_name="01_class_mean_change.png",
    zero_line=True,
)
display(class_change.sort_values(["metric", "mean"]))

## 2. Average change versus object attributes

Objects are divided into equal-frequency bins based on the attribute. Points show the mean within each bin and error bars show approximate 95% confidence intervals.

In [ ]:
change_by_mask_size = plot_attribute_results(
    df, "gt_mask_pixels", {"IoU": "delta_iou", "Dice": "delta_dice"},
    title="Performance change versus ground-truth mask size",
    xlabel="Median ground-truth mask size in bin (pixels; log scale)",
    ylabel="Mean perturbed score − original score",
    output_name="02_change_vs_mask_size.png", log_x=True, zero_line=True,
)

In [ ]:
change_by_coverage = plot_attribute_results(
    df, "mask_box_coverage", {"IoU": "delta_iou", "Dice": "delta_dice"},
    title="Performance change versus mask-to-box coverage",
    xlabel="Median fraction of bounding box covered by mask",
    ylabel="Mean perturbed score − original score",
    output_name="03_change_vs_mask_box_coverage.png", zero_line=True,
)

## 3. Image-level variance grouped by object class

For every image, variance is calculated across its perturbed-prompt scores. The plot then averages those image-level variances within each class.

In [ ]:
class_variance = plot_class_results(
    df,
    {"IoU": "variance_iou", "Dice": "variance_dice"},
    title="Prompt-induced image-level variance, by object class",
    xlabel="Mean within-image variance across perturbed prompts",
    output_name="04_class_prompt_variance.png",
)
display(class_variance.sort_values(["metric", "mean"], ascending=[True, False]))

## 4. Image-level variance versus object attributes

In [ ]:
variance_by_mask_size = plot_attribute_results(
    df, "gt_mask_pixels", {"IoU": "variance_iou", "Dice": "variance_dice"},
    title="Prompt-induced variance versus ground-truth mask size",
    xlabel="Median ground-truth mask size in bin (pixels; log scale)",
    ylabel="Mean within-image variance",
    output_name="05_variance_vs_mask_size.png", log_x=True,
)

In [ ]:
variance_by_coverage = plot_attribute_results(
    df, "mask_box_coverage", {"IoU": "variance_iou", "Dice": "variance_dice"},
    title="Prompt-induced variance versus mask-to-box coverage",
    xlabel="Median fraction of bounding box covered by mask",
    ylabel="Mean within-image variance",
    output_name="06_variance_vs_mask_box_coverage.png",
)

## 5. Optional: inspect the most sensitive images

This table is useful for connecting quantitative outliers to the overlay images generated earlier.

In [ ]:
sensitive = (
    df.assign(absolute_iou_change=df["delta_iou"].abs())
      .sort_values(["variance_iou", "absolute_iou_change"], ascending=False)
      [["image_id", "file_name", "annotation_id", "category_name",
        "original_iou", "mean_perturbed_iou", "delta_iou", "variance_iou",
        "gt_mask_pixels", "mask_box_coverage"]]
)
display(sensitive.head(20))
sensitive.to_csv(FIGURE_DIR / "most_sensitive_images.csv", index=False)